In [120]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [84]:
import sys
import os

sys.path.append(os.path.abspath('../'))

from tagging_system import automate_tagging as auto_tag
from models.custom_classifier import CustomClassifier

In [85]:
import numpy as np
import pandas as pd

In [86]:
books = pd.read_csv('../data/data_with_author_and_awards.csv',
dtype = {
    'isbn' : 'str', # do this explicitly to avoide getting a warning by the interpreter
    'author_birthyear' : 'Int64', # we have to explicitly do this to avoid pandas implicitly casting as float
    'title_id' : 'Int64'
},
)
books = books.dropna() # we drop the books without descriptions because these are very, very unlikely to be winners any

In [87]:
tags = auto_tag.load_tags()
transformer = auto_tag.load_transformer()
tags_encoded = auto_tag.encode_tags()
tags_map = auto_tag.tag_dictionary()

In [88]:
books = auto_tag.encode_books(books, transformer) # takes a while

In [91]:
books['target'] = books.hugo | books.locus
books['num_prev_awards'] = books.Hugo_Awards_Previously + books.Locus_Awards_Previously

books

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent,encoded_synopsis,target,num_prev_awards
0,9372,The Long Loud Silence,Wilson Tucker,1954,1954-00-00,Dell,1914,"Deer Creek, Illinois, USA",0899683754,Publisher's description: Tomorrow's war -- the...,...,40,0,0,False,False,USA,Central/North America,"[-0.041223828, 0.12155522, -0.006339091, 0.019...",False,0
4,1908,The Forgotten Planet,Murray Leinster,1954,1954-00-00,Ace Books,1896,"Norfolk, Virginia, USA",0881846163,"**From the first page of the Ace Double:** ""Na...",...,58,0,0,False,False,USA,Central/North America,"[-0.025720153, 0.054792713, -0.021301318, 0.03...",False,0
9,6101,The Star Beast,Robert A. Heinlein,1954,1954-08-23,Ace Books,1907,"Butler, Missouri, USA",0345275802,A talking alien pet has grown to the size of a...,...,47,0,0,False,False,USA,Central/North America,"[-0.03175946, 0.06456888, 0.05577806, -0.02333...",False,0
11,1066774,The Wheels of Chance,H. G. Wells,1954,1954-00-00,J. M. Dent,1866,"Bromley, Kent, England, UK",0460019147,The comical Wheels of Chance was written in 18...,...,88,0,0,False,False,UK,Europe,"[-0.0009298305, 0.06744729, 0.03826583, 0.0196...",False,0
17,2266,The Caves of Steel,Isaac Asimov,1954,1954-00-00,HarperCollins (UK),1920,"Petrovichi, Smolensk Governorate, Russia",0586008357,"""A Del Rey book."" It was bad enough when Lije ...",...,34,0,0,False,False,Russia,Europe,"[-0.11344511, 0.007374352, 0.012599569, 0.0339...",False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151563,3506858,Falling in a Sea of Stars,Kristen Britain,2025,2025-09-30,DAW Books,1965,"Batavia, New York, USA",9780756408824.0,"Magic, danger, and adventure abound for messen...",...,60,0,0,False,False,USA,Central/North America,"[-0.04759283, -0.034522656, -0.027196748, -0.0...",False,0
151566,3506902,Lover Forbidden,J. R. Ward,2025,2025-09-09,Gallery Books,1969,"Boston, Massachusetts, USA",9781982179960.0,The aristocracy is making a run for the throne...,...,56,0,0,False,False,USA,Central/North America,"[-0.08131112, -0.051669966, -0.076406844, 0.01...",False,0
151571,3507118,Fiend,Alma Katsu,2025,2025-09-16,G. P. Putnam's Sons,1959,"Fairbanks, Alaska, USA",9780593714348.0,"When Maris Berisha was nine years old, she hea...",...,66,0,0,False,False,USA,Central/North America,"[-0.044818047, 0.0097757345, 0.014525753, 0.02...",False,0
151579,3507075,Tiger's Trek,Colleen Houck,2025,2025-09-09,Blackstone Publishing,1969,"Tucson, Arizona, USA",9798212221733.0,From New York Times bestselling author Colleen...,...,56,0,0,False,False,USA,Central/North America,"[-0.041875966, -0.0069684396, 0.012509349, 0.1...",False,0


In [92]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score

In [94]:
# manually train test split our data

books_train = books[books.release_year <= 2014]
books_test = books[books.release_year <= 2014]

books_tt = books_train[books_train.release_year <= 2005]
books_val = books_train[books_train.release_year > 2005]

In [112]:
books_tt.head()

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent,encoded_synopsis,target,num_prev_awards
0,9372,The Long Loud Silence,Wilson Tucker,1954,1954-00-00,Dell,1914,"Deer Creek, Illinois, USA",0899683754,Publisher's description: Tomorrow's war -- the...,...,40,0,0,False,False,USA,Central/North America,"[-0.041223828, 0.12155522, -0.006339091, 0.019...",False,0
4,1908,The Forgotten Planet,Murray Leinster,1954,1954-00-00,Ace Books,1896,"Norfolk, Virginia, USA",0881846163,"**From the first page of the Ace Double:** ""Na...",...,58,0,0,False,False,USA,Central/North America,"[-0.025720153, 0.054792713, -0.021301318, 0.03...",False,0
9,6101,The Star Beast,Robert A. Heinlein,1954,1954-08-23,Ace Books,1907,"Butler, Missouri, USA",0345275802,A talking alien pet has grown to the size of a...,...,47,0,0,False,False,USA,Central/North America,"[-0.03175946, 0.06456888, 0.05577806, -0.02333...",False,0
11,1066774,The Wheels of Chance,H. G. Wells,1954,1954-00-00,J. M. Dent,1866,"Bromley, Kent, England, UK",0460019147,The comical Wheels of Chance was written in 18...,...,88,0,0,False,False,UK,Europe,"[-0.0009298305, 0.06744729, 0.03826583, 0.0196...",False,0
17,2266,The Caves of Steel,Isaac Asimov,1954,1954-00-00,HarperCollins (UK),1920,"Petrovichi, Smolensk Governorate, Russia",0586008357,"""A Del Rey book."" It was bad enough when Lije ...",...,34,0,0,False,False,Russia,Europe,"[-0.11344511, 0.007374352, 0.012599569, 0.0339...",False,0


In [146]:
#features = ['num_prev_awards', 'encoded_synopsis', 'first_publisher', 'author_birthplace_country', 'Author_Age_at_Publication']
features = ['Hugo_Awards_Previously', 'Locus_Awards_Previously', 'encoded_synopsis', 'first_publisher', 'author_birthplace_country', 'Author_Age_at_Publication']

In [147]:
X_tt = books_train[features]
y_tt = books_train['target']

X_val = books_val[features]
y_val = books_val['target']

In [149]:
my_classifier = CustomClassifier(n_neighbors = 2, weights='distance', n_jobs = -1, class_weight='balanced')

my_classifier.fit(X_tt, y_tt)

In [150]:
pred = my_classifier.predict(X_val)
print('f1', f1_score(y_val, pred))
print('precision', precision_score(y_val, pred))

f1 0.9724770642201835
precision 0.9532374100719424


These are some suspiciously good metrics...

In [151]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_val, pred, normalize='all')

array([[9.57068384e-01, 1.99325360e-03],
       [3.06654400e-04, 4.06317081e-02]])

It looks like what's happening is that it's making any false negatives or false positives, but have a decently high number of true positives, and a very high rate of true negatives.

In [152]:
tt_cv = [ books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 1977)] ]
val_cv = [ books_train[(1978 <= books_train.release_year) & (books_train.release_year <= 1979)] ]

tt_cv.append( books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 1985)] )
val_cv.append( books_train[(1986 <= books_train.release_year) & (books_train.release_year <= 1987)] )

tt_cv.append( books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 1993)] )
val_cv.append( books_train[(1994 <= books_train.release_year) & (books_train.release_year <= 1995)] )

tt_cv.append( books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 2001)] )
val_cv.append( books_train[(2002 <= books_train.release_year) & (books_train.release_year <= 2003)] )

tt_cv.append( books_train[(1971 <= books_train.release_year) & (books_train.release_year <= 2011)] )
val_cv.append( books_train[(2012 <= books_train.release_year) & (books_train.release_year <= 2014)] )

In [153]:
f1s = dict()
precs = dict()

my_classifier = CustomClassifier(n_neighbors = 2, weights='distance', n_jobs = -1, class_weight='balanced')

for i in range(5):
    X_tt_cv = tt_cv[i][features]
    y_tt_cv = tt_cv[i].target

    X_ho = val_cv[i][features]
    y_ho = val_cv[i].target

    my_classifier.fit(X_tt_cv, y_tt_cv)

    pred = my_classifier.predict(X_ho)

    f1s['Fold {} F_1'.format(i)] = f1_score(y_ho, pred)
    precs['Fold {} Precision'.format(i)] = precision_score(y_ho, pred)
    

In [154]:
avg = 0
for key in f1s:
    print(key, f1s[key], sep=": ")
    avg += f1s[key]

print(avg / 5)


Fold 0 F_1: 0.2191780821917808
Fold 1 F_1: 0.2204724409448819
Fold 2 F_1: 0.23529411764705882
Fold 3 F_1: 0.17177914110429449
Fold 4 F_1: 0.16988416988416988
0.2033215903544372


In [155]:
avg = 0
for key in precs:
    print(key, precs[key], sep=": ")
    avg += precs[key]

print(avg / 5)

Fold 0 Precision: 0.22857142857142856
Fold 1 Precision: 0.20588235294117646
Fold 2 Precision: 0.1836734693877551
Fold 3 Precision: 0.1320754716981132
Fold 4 Precision: 0.14102564102564102
0.17824567272482286


As expected, we get much lower scores when we do cross-validation. This makes me wonder what was happening when we did the fitting on books_tt to get such ridiculously high scores.

In [ ]:
from models.custom_classifier import CustomBaggingClassifier

n_ests = [2, 5, 10, 100, 1000]

avg_f1s = dict()
avg_precs = dict()

kwargs = {'n_neighbors' : 2, 'weights' : 'distance', 'n_jobs' : -1, 'class_weight' :'balanced'}


for n in n_ests:
    avg_f1 = 0
    avg_prec = 0
    bag = CustomBaggingClassifier(
            base_estimator = CustomClassifier,
            n_estimators = n,
            kwargs = kwargs
        )
    for i in range(5):

        X_tt_cv = tt_cv[i][features]
        y_tt_cv = tt_cv[i].target

        X_ho = val_cv[i][features]
        y_ho = val_cv[i].target

        bag.fit(X_tt_cv, y_tt_cv)
        pred = bag.predict(X_ho)

        avg_f1 += f1_score(y_ho, pred)
        avg_prec += precision_score(y_ho, pred)

    avg_f1s['{} estimators'.format(n)] = avg_f1 / 5
    avg_precs['{} estimators'.format(n)] = avg_prec / 5

In [158]:
avg_f1s

{'2 estimators': 0.17967522193895585,
 '5 estimators': 0.18746573514785436,
 '10 estimators': 0.18368488281866543,
 '100 estimators': 0.20145432594643617}

In [159]:
avg_precs

{'2 estimators': 0.20586946734755185,
 '5 estimators': 0.1728224431267682,
 '10 estimators': 0.17194137931034484,
 '100 estimators': 0.17789145982498805}

It seems on this sample, bagging with 5 estimators seem to do the best in terms of $F_1$ and precision.